In [4]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score, confusion_matrix, classification_report

In [5]:

# after the 100 runs results (csv)
a = pd.read_csv('cancel_tuning_results.csv')
b = pd.read_csv('delay_tuning_results.csv')


print('Model A shape:', a.shape)
print('Model B shape:', b.shape)

Model A shape: (100, 15)
Model B shape: (100, 15)


In [6]:
# best trials for each model 
best_a = a.iloc[0]
best_b = b.iloc[0]

print('Best Model A trial:')
display(best_a.to_frame('value'))

print('Best Model B trial:')
display(best_b.to_frame('value'))

Best Model A trial:


,value
task,cancel
trial,91
val_pr_auc,0.134856
val_roc_auc,0.760726
val_best_f1,0.192396
val_best_threshold,0.73
best_iteration,16
max_depth,4
min_child_weight,10.0
learning_rate,0.1


Best Model B trial:


,value
task,delay_non_cancelled
trial,32
val_pr_auc,0.36397
val_roc_auc,0.70784
val_best_f1,0.4014
val_best_threshold,0.53
best_iteration,93
max_depth,9
min_child_weight,1.0
learning_rate,0.03


In [7]:
# run the best results on models
df = pd.read_parquet('targets_features_split.parquet')

# col groups
id_cols = ['FlightDate', 'split']
target_cols = ['target', 'target_cancelled', 'target_delayed', 'target_delayed_non_cancelled']
feature_cols = [c for c in df.columns if c not in set(id_cols + target_cols)]

# best threshold
thr_a = float(best_a['val_best_threshold'])
thr_b = float(best_b['val_best_threshold'])

print('Best threshold-Model A:', thr_a)
print('Best threshold-Model B:', thr_b)
print('Feature count:', len(feature_cols))



Best threshold-Model A: 0.73
Best threshold-Model B: 0.53
Feature count: 35


In [8]:

def get_best_params(row):
    return {
        'max_depth': int(row['max_depth']),
        'min_child_weight': float(row['min_child_weight']),
        'learning_rate': float(row['learning_rate']),
        'subsample': float(row['subsample']),
        'colsample_bytree': float(row['colsample_bytree']),
        'gamma': float(row['gamma']),
        'reg_alpha': float(row['reg_alpha']),
        'reg_lambda': float(row['reg_lambda']),
        
    }


def compute_class_weight_ratio(y):
    y = np.asarray(y).astype(int)
    pos = int(y.sum())
    neg = int((1-y).sum())
    return neg / max(pos, 1)

def print_metrics(name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    print(f'\n{name}')
    print(f'ROC-AUC: {roc_auc_score(y_true, y_prob):.4f}')
    print(f'PR-AUC: {average_precision_score(y_true, y_prob):.4f}')
    print(f'F1: {f1_score(y_true, y_pred):.4f}')
    print(f'Precision: {precision_score(y_true, y_pred):.4f}')
    print(f'Recall: {recall_score(y_true, y_pred):.4f}')
    print(f'Accuracy: {accuracy_score(y_true, y_pred):.4f}')
    print('Confusion matrix:')
    print(confusion_matrix(y_true, y_pred))

# split masks and data
train = df['split'] == 'train'
val = df['split'] == 'val'
test = df['split'] == 'test'

In [ ]:
# Model A
X_train_A = df.loc[train, feature_cols]
X_val_A = df.loc[val, feature_cols]
X_test_A = df.loc[test, feature_cols]

y_train_A = df.loc[train, 'target_cancelled'].astype(int)
y_val_A = df.loc[val, 'target_cancelled'].astype(int)
y_test_A = df.loc[test, 'target_cancelled'].astype(int)

params_a = get_best_params(best_a)

model_A = XGBClassifier(
    n_estimators = 3000,
    objective='binary:logistic',
    eval_metric='aucpr',
    tree_method='hist',
    n_jobs=-1,
    random_state=42,
    scale_pos_weight=compute_class_weight_ratio(y_train_A),
    **params_a
)
model_A.fit(X_train_A, y_train_A, eval_set=[(X_val_A, y_val_A)], verbose=False)

p_train_A = model_A.predict_proba(X_train_A)[:,1]
p_val_A = model_A.predict_proba(X_val_A)[:,1]
p_test_A = model_A.predict_proba(X_test_A)[:,1]

print('Model A:')
print_metrics('(Train):', y_train_A, p_train_A, threshold=thr_a)
print_metrics('(Val):', y_val_A, p_val_A, threshold=thr_a)
print_metrics('(Test):', y_test_A, p_test_A, threshold=thr_a)

# Model B
train_B = train & (df['target_delayed_non_cancelled'] >= 0)
val_B = val & (df['target_delayed_non_cancelled'] >= 0)
test_B = test & (df['target_delayed_non_cancelled'] >= 0)

X_train_B = df.loc[train_B, feature_cols]
X_val_B = df.loc[val_B, feature_cols]
X_test_B = df.loc[test_B, feature_cols]

y_train_B = df.loc[train_B, 'target_delayed_non_cancelled'].astype(int)
y_val_B = df.loc[val_B, 'target_delayed_non_cancelled'].astype(int)
y_test_B = df.loc[test_B, 'target_delayed_non_cancelled'].astype(int)

params_b = get_params_from_row(best_b)

model_B = XGBClassifier(
    n_estimators=3000,
    objective='binary:logistic',
    eval_metric='aucpr',
    tree_method='hist',
    n_jobs=-1,
    random_state=49,
    scale_pos_weight=compute_class_weight_ratio(y_train_B),
    **params_b
)

model_B.fit(X_train_B, y_train_B, eval_set=[(X_val_B, y_val_B)], verbose=False)

p_train_B = model_B.predict_proba(X_train_B)[:, 1]
p_val_B = model_B.predict_proba(X_val_B)[:, 1]
p_test_B = model_B.predict_proba(X_test_B)[:, 1]

print_metrics('Model B - Train', y_train_B, p_train_B, threshold=thr_b)
print_metrics('Model B - Validation', y_val_B, p_val_B, threshold=thr_b)
print_metrics('Model B - Test', y_test_B, p_test_B, threshold=thr_b)

# cascade of tuned models 
# Model B probabilities on full val/test rows for cascade combine
X_val_full = df.loc[val, feature_cols]
X_test_full = df.loc[test, feature_cols]

p_val_B_full = model_B.predict_proba(X_val_full)[:, 1]
p_test_B_full = model_B.predict_proba(X_test_full)[:, 1]

y_val_multi = df.loc[val, 'target'].astype(int).values
y_test_multi = df.loc[test, 'target'].astype(int).values

# if A predicts cancelled => class 2, else use B (0/1)
y_pred_val_multi = np.where((p_val_A >= thr_a).astype(int) == 1, 2, (p_val_B_full >= thr_b).astype(int))
y_pred_test_multi = np.where((p_test_A >= thr_a).astype(int) == 1, 2, (p_test_B_full >= thr_b).astype(int))

print('\nCascade:')
print('(Validation)\n')
print('Accuracy:', accuracy_score(y_val_multi, y_pred_val_multi))
print('Confusion matrix:')
print(confusion_matrix(y_val_multi, y_pred_val_multi))
print(classification_report(y_val_multi, y_pred_val_multi, digits=4))

print('\nCascade')
print('(Test)\n')
print('Accuracy:', accuracy_score(y_test_multi, y_pred_test_multi))
print('Confusion matrix:')
print(confusion_matrix(y_test_multi, y_pred_test_multi))
print(classification_report(y_test_multi, y_pred_test_multi, digits=4))

